# FAISS Semantic Search: Dense Vector Concept Mapping

This notebook demonstrates how to use **FAISS (Facebook AI Similarity Search)** as the
knowledge layer backend for OMOP CDM concept mapping with the Portiere SDK.

Unlike lexical search (BM25s), FAISS performs **dense vector similarity search** --
each clinical term is encoded into a high-dimensional vector using a biomedical language
model (SapBERT), and retrieval is based on vector cosine similarity. This means FAISS
can find semantically related concepts even when the exact words differ.

For example, the ICD-10 description "Type 2 diabetes mellitus without complications"
will match the SNOMED concept "Type 2 diabetes mellitus" because the dense vectors
capture the shared clinical meaning -- even though the strings are not identical.

## What We Will Do

1. **Install** Portiere with FAISS support
2. **Build indexes** from the OHDSI Athena vocabulary (SNOMED, LOINC, RxNorm, ICD10CM) in `vocabulary_download_v5/`
3. **Configure** Portiere with the FAISS backend
4. **Create a project** targeting OMOP CDM v5.4
5. **Add data sources** -- patients (demographics) and diagnoses (ICD-10 codes)
6. **Map schema** -- automatically map source columns to OMOP CDM tables
7. **Map concepts via FAISS** -- use dense vector search to match diagnosis codes to standard concepts
8. **Inspect semantic matches** -- see how FAISS handles synonyms and abbreviations
9. **Run ETL** -- generate transformed OMOP CDM output files

## Lexical vs. Semantic Search

| Feature | BM25s (Lexical) | FAISS (Semantic) |
|---------|-----------------|------------------|
| Search method | Keyword/token matching | Dense vector similarity |
| Handles synonyms | Limited | Strong |
| Handles abbreviations | Poor | Good |
| Speed | Fast | Fast (after index build) |
| Dependencies | None (built-in) | sentence-transformers, faiss-cpu |
| Best for | Exact code lookups | Clinical term matching |

## Prerequisites

- Python 3.10+
- Two CSV data sources: `patients.csv` (15 patients) and `diagnoses.csv` (20 diagnosis records)
- OHDSI Athena vocabulary (SNOMED, LOINC, RxNorm, ICD10CM) from `vocabulary_download_v5/` -- indexes are built automatically on first run

## Pipeline Overview

**Ingest** --> **Profile** --> **Schema Map** --> **ETL Gen** --> **Validate**

In this notebook, the concept mapping stage uses FAISS dense vector retrieval instead of
BM25s keyword matching. All other stages remain the same as the local mode quickstart.

## Step 1: Install Portiere with FAISS Support

Install the Portiere SDK with the `faiss` extra. This pulls in `faiss-cpu` and
`sentence-transformers` as additional dependencies.

In [1]:
!pip install portiere-health[faiss]
!pip install sentence-transformers


zsh:1: no matches found: portiere[faiss]



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install sentence_transformers


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Step 2: Build Indexes from Athena Vocabulary

Before we can search, we need to build FAISS and BM25s indexes from the OHDSI Athena
vocabulary. This encodes each concept name into a dense vector using SapBERT and stores
the vectors in a FAISS index for fast similarity search.

This step runs once on first execution (~10-30 min for FAISS embedding). On subsequent
runs, the existing indexes are reused from disk.

In [3]:
from pathlib import Path
from portiere.knowledge import build_knowledge_layer

ATHENA_DIR = Path("example_assets/vocabulary_download_v5")
VOCAB_DIR  = ATHENA_DIR / "indexes"
BM25S_CORPUS = VOCAB_DIR / "concepts.json"
FAISS_INDEX  = VOCAB_DIR / "concepts.index"
FAISS_META   = VOCAB_DIR / "concepts.meta.json"

# Build indexes from Athena on first run (~10-30 min for FAISS embedding)
if not BM25S_CORPUS.exists():
    paths = build_knowledge_layer(
        athena_path=ATHENA_DIR,
        output_path=VOCAB_DIR,
        backend="hybrid",
        vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    )
    print(f"Built indexes: {paths}")
else:
    print(f"Using existing indexes at {VOCAB_DIR}")

Using existing indexes at example_assets/vocabulary_download_v5/indexes


## Step 3: Import and Configure

We configure Portiere with the **FAISS backend** for dense vector concept retrieval.

Portiere uses **intent-based configuration** -- it infers the operating mode automatically.
When a `knowledge_layer` is configured and no `api_key` is provided, Portiere infers
local mode. There is no need to set `mode` or `pipeline` explicitly.

Key settings:

- `knowledge_layer` with `backend="faiss"` -- uses FAISS for dense vector retrieval
- `faiss_index_path` / `faiss_metadata_path` -- paths to the FAISS index built in Step 2
- `embedding` with SapBERT -- biomedical embedding model for encoding clinical terms
- `reranker` with `provider="none"` -- no reranker, pure FAISS retrieval

In [4]:
from pathlib import Path

ATHENA_DIR = Path("example_assets/vocabulary_download_v5")
VOCAB_DIR  = ATHENA_DIR / "indexes"
BM25S_CORPUS = VOCAB_DIR / "concepts.json"
FAISS_INDEX  = VOCAB_DIR / "concepts.index"
FAISS_META   = VOCAB_DIR / "concepts.meta.json"

In [5]:
import portiere
from portiere import PortiereConfig, KnowledgeLayerConfig
from portiere.config import EmbeddingConfig, RerankerConfig

config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="faiss",                   # Use FAISS dense vector search
        faiss_index_path=str(FAISS_INDEX),
        faiss_metadata_path=str(FAISS_META),
    ),
    embedding=EmbeddingConfig(
        provider="huggingface",            # Load model from HuggingFace
        model="cambridgeltl/SapBERT-from-PubMedBERT-fulltext",  # Biomedical embeddings
    ),
    reranker=RerankerConfig(provider="none"),  # No reranker -- pure FAISS
)

print(f"Mode: {config.effective_mode}")
print(f"Pipeline: {config.effective_pipeline}")
print(f"Knowledge backend: {config.knowledge_layer.backend}")
print(f"Embedding model: {config.embedding.model}")

Mode: local
Pipeline: local
Knowledge backend: faiss
Embedding model: cambridgeltl/SapBERT-from-PubMedBERT-fulltext


## Step 4: Create a Project

Initialize a new mapping project targeting OMOP CDM v5.4 with standard vocabularies.
The project uses the Polars engine for fast local data processing.

In [6]:
from portiere.engines import PolarsEngine

project = portiere.init(
    name="FAISS Semantic Search Demo",
    engine=PolarsEngine(),
    target_model="omop_cdm_v5.4",
    vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    config=config,
)

print(project)

2026-04-16 00:30:42 [info     ] PolarsEngine initialized      


2026-04-16 00:30:42 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


Project(name='FAISS Semantic Search Demo', task='standardize', target_model='omop_cdm_v5.4', mode='local')


## Step 5: Add Data Sources

Add two CSV files as data sources:

- **patients.csv** -- 15 patients with 11 demographic columns (patient_id, name, DOB, gender, race, etc.)
- **diagnoses.csv** -- 20 diagnosis records with ICD-10 codes (E11.9, I10, J06.9, J44.1, N18.3, R51, F32.9, Z87.891)

The SDK reads each file, detects its format, and profiles the data for schema mapping.

In [7]:
patients_source = project.add_source(
    "example_assets/01.2_faiss_semantic_search/patients.csv"
)
diagnoses_source = project.add_source(
    "example_assets/01.2_faiss_semantic_search/diagnoses.csv"
)

print(f"Patients source: {patients_source['name']}, format: {patients_source['format']}")
print(f"  Columns: {patients_source.get('columns', 'N/A')}")
print(f"  Rows: {patients_source.get('row_count', 'N/A')}")
print()
print(f"Diagnoses source: {diagnoses_source['name']}, format: {diagnoses_source['format']}")
print(f"  Columns: {diagnoses_source.get('columns', 'N/A')}")
print(f"  Rows: {diagnoses_source.get('row_count', 'N/A')}")

2026-04-16 00:30:42 [info     ] project.source_added           project='FAISS Semantic Search Demo' source=patients


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:30:45 [info     ] gx_profiler.profiled           columns=11 rows=15 source=patients


2026-04-16 00:30:45 [info     ] project.profiled               source=patients


2026-04-16 00:30:45 [info     ] project.source_added           project='FAISS Semantic Search Demo' source=diagnoses


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:30:45 [info     ] gx_profiler.profiled           columns=6 rows=20 source=diagnoses


2026-04-16 00:30:45 [info     ] project.profiled               source=diagnoses


Patients source: patients, format: csv
  Columns: ['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'ethnicity', 'address', 'city', 'state', 'zip_code']
  Rows: 15

Diagnoses source: diagnoses, format: csv
  Columns: ['encounter_id', 'patient_id', 'diagnosis_code', 'diagnosis_description', 'diagnosis_type', 'diagnosis_date']
  Rows: 20


## Step 6: Map Schema

Schema mapping automatically matches source columns to OMOP CDM target tables and columns.
We map schema for both data sources.

Confidence routing determines the status of each mapping:

| Confidence Range | Status | Action |
|-----------------|--------|--------|
| >= 0.95 | AUTO_ACCEPTED | No review needed |
| 0.80 - 0.95 | VERIFY | Quick verification recommended |
| 0.70 - 0.80 | REVIEW | Manual review needed |
| < 0.70 | MANUAL | Requires manual mapping |

In [8]:
patients_schema_map = project.map_schema(patients_source)
diagnoses_schema_map = project.map_schema(diagnoses_source)

print("Patients schema mapping:")
print(f"  Summary: {patients_schema_map.summary()}")
print()
print("Diagnoses schema mapping:")
print(f"  Summary: {diagnoses_schema_map.summary()}")

2026-04-16 00:30:45 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=omop_cdm_v5.4


2026-04-16 00:30:45 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:30:45 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:30:45 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:30:47 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


...

2026-04-16 00:30:47 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:30:47 [info     ] Stage 2 complete               auto=5 review=0 total=6


2026-04-16 00:30:47 [info     ] local_storage.schema_mapping_saved items_count=6 project='FAISS Semantic Search Demo'


2026-04-16 00:30:47 [info     ] project.schema_mapped          auto=5 project='FAISS Semantic Search Demo' total=6


Patients schema mapping:
  Summary: {'total': 11, 'auto_accepted': 10, 'needs_review': 0, 'approved': 0, 'unmapped': 1, 'auto_rate': 90.9090909090909}

Diagnoses schema mapping:
  Summary: {'total': 6, 'auto_accepted': 5, 'needs_review': 0, 'approved': 0, 'unmapped': 1, 'auto_rate': 83.33333333333334}


In [9]:
# Export to DataFrame
patients_df = patients_schema_map.to_dataframe()
print("Patients Schema mapping as DataFrame:")
patients_df

Patients Schema mapping as DataFrame:


source_column,source_table,target_table,target_column,confidence,status
str,str,str,str,f64,str
"""patient_id""","""""","""person""","""person_id""",0.95,"""auto_accepted"""
"""first_name""","""""","""person""","""person_source_value""",0.95,"""auto_accepted"""
"""last_name""","""""","""person""","""person_source_value""",0.5,"""unmapped"""
"""date_of_birth""","""""","""person""","""birth_datetime""",0.95,"""auto_accepted"""
"""gender""","""""","""person""","""gender_concept_id""",0.95,"""auto_accepted"""
…,…,…,…,…,…
"""ethnicity""","""""","""person""","""ethnicity_concept_id""",0.95,"""auto_accepted"""
"""address""","""""","""location""","""address_1""",0.95,"""auto_accepted"""
"""city""","""""","""location""","""city""",0.95,"""auto_accepted"""


In [10]:
# Export to DataFrame
diagnoses_df = diagnoses_schema_map.to_dataframe()
print("Diagnoses Schema mapping as DataFrame:")
diagnoses_df

Diagnoses Schema mapping as DataFrame:


source_column,source_table,target_table,target_column,confidence,status
str,str,str,str,f64,str
"""encounter_id""","""""","""visit_occurrence""","""visit_occurrence_id""",0.95,"""auto_accepted"""
"""patient_id""","""""","""person""","""person_id""",0.95,"""auto_accepted"""
"""diagnosis_code""","""""","""condition_occurrence""","""condition_source_value""",0.95,"""auto_accepted"""
"""diagnosis_description""","""""","""condition_occurrence""","""condition_source_value""",0.5,"""unmapped"""
"""diagnosis_type""","""""","""condition_occurrence""","""condition_type_concept_id""",0.95,"""auto_accepted"""
"""diagnosis_date""","""""","""condition_occurrence""","""condition_start_date""",0.95,"""auto_accepted"""


## Step 7: Review Schema Mappings

Let's inspect the individual mapping items for both sources. Each item shows:

- **source_column** -- the original column name from the CSV
- **target_table** / **target_column** -- the mapped OMOP CDM destination
- **confidence** -- how confident the algorithm is in this mapping (0.0 to 1.0)
- **status** -- the current mapping status

In [11]:
print("Patients Schema Mapping Results:")
print("-" * 80)
for item in patients_schema_map.items:
    print(
        f"  {item.source_column} -> {item.target_table}.{item.target_column} "
        f"(confidence: {item.confidence:.2f}, status: {item.status.value})"
    )

print()
print("Diagnoses Schema Mapping Results:")
print("-" * 80)
for item in diagnoses_schema_map.items:
    print(
        f"  {item.source_column} -> {item.target_table}.{item.target_column} "
        f"(confidence: {item.confidence:.2f}, status: {item.status.value})"
    )

Patients Schema Mapping Results:
--------------------------------------------------------------------------------
  patient_id -> person.person_id (confidence: 0.95, status: auto_accepted)
  first_name -> person.person_source_value (confidence: 0.95, status: auto_accepted)
  last_name -> person.person_source_value (confidence: 0.50, status: unmapped)
  date_of_birth -> person.birth_datetime (confidence: 0.95, status: auto_accepted)
  gender -> person.gender_concept_id (confidence: 0.95, status: auto_accepted)
  race -> person.race_concept_id (confidence: 0.95, status: auto_accepted)
  ethnicity -> person.ethnicity_concept_id (confidence: 0.95, status: auto_accepted)
  address -> location.address_1 (confidence: 0.95, status: auto_accepted)
  city -> location.city (confidence: 0.95, status: auto_accepted)
  state -> location.state (confidence: 0.95, status: auto_accepted)
  zip_code -> location.zip (confidence: 0.95, status: auto_accepted)

Diagnoses Schema Mapping Results:
-------------

## Step 8: Approve Schema Mappings

After reviewing, approve all mappings and finalize both schema mappings.

- `approve_all()` -- marks all remaining items as approved
- `finalize()` -- locks the schema mapping so it can be used in ETL generation

In [12]:
patients_schema_map.approve_all()
patients_schema_map.finalize()

diagnoses_schema_map.approve_all()
diagnoses_schema_map.finalize()

print("Schema mappings finalized.")
print(f"Patients summary: {patients_schema_map.summary()}")
print(f"Diagnoses summary: {diagnoses_schema_map.summary()}")

Schema mappings finalized.
Patients summary: {'total': 11, 'auto_accepted': 10, 'needs_review': 0, 'approved': 0, 'unmapped': 1, 'auto_rate': 90.9090909090909}
Diagnoses summary: {'total': 6, 'auto_accepted': 5, 'needs_review': 0, 'approved': 0, 'unmapped': 1, 'auto_rate': 83.33333333333334}


## Step 9: Map Concepts via FAISS

This is the key section. We use **FAISS semantic search** to map diagnosis codes from
our `diagnoses.csv` to standard OMOP concepts.

When you pass `source=diagnoses_source`, Portiere auto-detects columns likely to contain
clinical codes (e.g., `diagnosis_code`) and maps their values using the FAISS index.

Under the hood:
1. Each source code description is encoded into a dense vector using SapBERT
2. FAISS performs approximate nearest-neighbor search against the concept index
3. The closest concept vectors are returned with similarity scores
4. Confidence routing assigns each mapping a status (AUTO, VERIFY, REVIEW, MANUAL)

In [13]:
concept_map = project.map_concepts(source=diagnoses_source)

print(concept_map)
print(f"\nSummary: {concept_map.summary()}")

2026-04-16 00:30:47 [info     ] project.code_columns_detected  columns=['diagnosis_code']


2026-04-16 00:30:47 [info     ] Stage 3: Concept mapping (local) columns=['diagnosis_code'] knowledge_backend=faiss


2026-04-16 00:30:47 [info     ] Mapping column: diagnosis_code


2026-04-16 00:30:47 [info     ] Found description column: diagnosis_description code_column=diagnosis_code desc_column=diagnosis_description mappings=8


2026-04-16 00:30:47 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:30:47 [info     ] knowledge.creating_backend     backend=faiss model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext


/Users/kuangsmacbook/Downloads/CUSP/PORTIERE/portiere/src/portiere/engines/polars_engine.py:131: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


2026-04-16 00:30:50 [info     ] faiss.index_loaded             concepts_count=623910 index_size=623910


2026-04-16 00:30:50 [info     ] local_concept_mapper.initialized code_index_entries=0 knowledge_backend=LocalFAISSBackend llm_verifier=False reranker=False


ImportError: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers

## Step 10: Inspect Individual Mappings

Let's examine each concept mapping result. Each item includes:

- **source_code** -- the original ICD-10 code from the diagnoses file
- **target_concept_name** -- the matched standard concept name (SNOMED, ICD10CM, etc.)
- **method** -- how the mapping was determined (AUTO, VERIFY, MANUAL, OVERRIDE)
- **confidence** -- the FAISS similarity score

Notice how FAISS matches semantically related terms. For example:
- "Type 2 diabetes mellitus without complications" (ICD-10 E11.9) finds "Type 2 diabetes mellitus" (SNOMED)
- "Essential (primary) hypertension" (ICD-10 I10) finds "Essential hypertension" (SNOMED)
- "Chronic obstructive pulmonary disease with acute exacerbation" (ICD-10 J44.1) finds "Chronic obstructive lung disease" (SNOMED)

These matches work because the dense vectors capture the shared clinical meaning,
even when the exact wording differs between ICD-10 and SNOMED descriptions.

In [14]:
print("Concept Mapping Results:")
print("-" * 80)

for item in concept_map.items:
    print(
        f"Source Code = {item.source_code}\n"
        f"Source Name = {item.source_description}\n" 
        f"Target Code: {item.target_vocabulary_id} | {item.target_concept_id}\n"
        f"Target Name: {item.target_concept_name}\n"
        f"Domain: {item.target_domain_id}\n"
        f"({item.method.value}, confidence: {item.confidence:.2f})\n\n"
    )

Concept Mapping Results:
--------------------------------------------------------------------------------


NameError: name 'concept_map' is not defined

### Check Concept Mappings Needing Review

The `needs_review()` method returns only the items that require human attention.

In [15]:
review_concepts = concept_map.needs_review()

print(f"Concept mappings needing review: {len(review_concepts)}")
for item in review_concepts:
    print(f"  {item.source_code}: {item.target_concept_name} (confidence: {item.confidence:.2f})")

NameError: name 'concept_map' is not defined

### Check Concept Mappings Auto-mapped

The `auto_mapped()` method.

In [16]:
auto_mapped_concepts = concept_map.auto_mapped()

print(f"Concept mappings auto mapped: {len(auto_mapped_concepts)}")
for item in auto_mapped_concepts:
    print(f"  {item.source_code}: {item.target_concept_name} (confidence: {item.confidence:.2f})")

NameError: name 'concept_map' is not defined

## Step 11: Semantic Matching Advantage

Let's demonstrate FAISS's key advantage: **semantic similarity matching**.

We pass explicit ICD-10 codes whose descriptions differ from the SNOMED concept names
in the index. A keyword-based search might struggle with these because the wording
is not identical, but FAISS finds the correct match through vector similarity.

Examples of cross-vocabulary matching:

| ICD-10 Code | ICD-10 Description | Expected SNOMED Match |
|-------------|--------------------|-----------------------|
| E11.9 | Type 2 diabetes mellitus without complications | Type 2 diabetes mellitus |
| I10 | Essential (primary) hypertension | Essential hypertension |
| J44.1 | COPD with acute exacerbation | Chronic obstructive lung disease |
| R51 | Headache | Headache |
| F32.9 | Major depressive disorder single episode unspecified | Major depressive disorder |

In [17]:
# Map explicit codes that have different wording than the SNOMED index entries
synonym_map = project.map_concepts(
    codes=["E11.9", "I10", "J44.1", "R51", "F32.9"]
)

print("Semantic Matching Results (ICD-10 -> SNOMED via FAISS):")
print("-" * 80)

for item in synonym_map.items:
    print(
        f"  {item.source_code}: {item.target_concept_name} "
        f"(confidence: {item.confidence:.2f})"
    )

print(f"\nSummary: {synonym_map.summary()}")
print(
    "\nFAISS matches these codes based on the semantic similarity of their descriptions,"
    "\nnot by exact string matching. This is why 'Type 2 diabetes mellitus without"
    "\ncomplications' correctly maps to the SNOMED concept 'Type 2 diabetes mellitus'"
    "\neven though the strings differ."
)

2026-04-16 00:30:50 [info     ] Stage 3: Concept mapping (local) columns=[] knowledge_backend=faiss


2026-04-16 00:30:50 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:30:50 [info     ] knowledge.creating_backend     backend=faiss model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext


2026-04-16 00:30:54 [info     ] faiss.index_loaded             concepts_count=623910 index_size=623910


2026-04-16 00:30:54 [info     ] local_concept_mapper.initialized code_index_entries=0 knowledge_backend=LocalFAISSBackend llm_verifier=False reranker=False


ImportError: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers

## Step 12: Approve and Finalize Concepts

After reviewing the FAISS-powered concept mappings, approve all and finalize.

In [18]:
concept_map.approve_all()

print("Concept mappings approved.")
print(f"Updated summary: {concept_map.summary()}")

NameError: name 'concept_map' is not defined

## Step 13: Run ETL

Generate the transformed output. The ETL step takes the finalized schema and concept
mappings and produces OMOP CDM-compliant output files.

We run ETL for both data sources:

- **patients** -- demographic data mapped to `person` and `location` tables
- **diagnoses** -- diagnosis codes mapped to `condition_occurrence` with standard concepts

In [19]:
patients_etl = project.run_etl(
    patients_source,
    output_dir="./output_faiss",
    schema_mapping=patients_schema_map,
    concept_mapping=concept_map,
)

print(f"Patients ETL complete: {patients_etl}")

NameError: name 'concept_map' is not defined

In [20]:
diagnoses_etl = project.run_etl(
    diagnoses_source,
    output_dir="./output_faiss",
    schema_mapping=diagnoses_schema_map,
    concept_mapping=concept_map,
)

print(f"Diagnoses ETL complete: {diagnoses_etl}")

NameError: name 'concept_map' is not defined

### Validate Output (Optional)

Run validation checks on the ETL output to ensure data quality and target model conformance.

In [21]:
patients_validation = project.validate(etl_result=patients_etl)
print(f"Patients validation: all_passed={patients_validation.get('all_passed', 'N/A')}")

for table in patients_validation.get("tables", []):
    print(
        f"  {table['table_name']}: passed={table['passed']}, "
        f"completeness={table['completeness_score']:.2f}, "
        f"conformance={table['conformance_score']:.2f}"
    )

NameError: name 'patients_etl' is not defined

In [22]:
diagnoses_validation = project.validate(etl_result=diagnoses_etl)
print(f"Diagnoses validation: all_passed={diagnoses_validation.get('all_passed', 'N/A')}")

for table in diagnoses_validation.get("tables", []):
    print(
        f"  {table['table_name']}: passed={table['passed']}, "
        f"completeness={table['completeness_score']:.2f}, "
        f"conformance={table['conformance_score']:.2f}"
    )

NameError: name 'diagnoses_etl' is not defined

---

## Summary

In this notebook we completed a full end-to-end OMOP CDM mapping workflow using
**FAISS semantic search** as the knowledge layer backend:

1. **SapBERT embeddings** -- The `cambridgeltl/SapBERT-from-PubMedBERT-fulltext` model
   encodes clinical terms into dense vectors that capture biomedical meaning. This is
   purpose-built for medical terminology, outperforming general-purpose embeddings.

2. **Dense vector similarity** -- FAISS finds the closest concept by vector cosine
   similarity, which captures meaning beyond keywords. "Type 2 diabetes mellitus
   without complications" matches "Type 2 diabetes mellitus" because the vectors
   are semantically close, even though the strings differ.

3. **Cross-vocabulary matching** -- FAISS handles the mapping between ICD-10-CM
   descriptions and SNOMED concept names naturally, because the embedding model
   understands that different phrasings of the same clinical concept are related.

4. **Dependencies** -- Requires `sentence-transformers` and `faiss-cpu` (installed
   via `pip install portiere-health[faiss]`). The FAISS index must be prebuilt with the
   same embedding model used at query time.

5. **Pure retrieval** -- This notebook used no reranker (`provider="none"`). For
   even better accuracy, add a cross-encoder reranker on top of FAISS retrieval.

## Next Steps

- **Hybrid + Reranking**: See `01.3_hybrid_search_reranking.ipynb` to combine FAISS (dense)
  with BM25s (sparse) using Reciprocal Rank Fusion (RRF), plus a cross-encoder reranker
  for maximum accuracy.
- **LLM-Verified Mapping**: See `01.4_llm_verified_mapping.ipynb` to add LLM verification
  for borderline-confidence mappings.
- **Cloud pipeline**: See `02_cloud_pipeline.ipynb` to use Portiere's cloud AI inference.
- **BYO-LLM**: See `03_byo_llm_providers.ipynb` to use your own LLM provider (OpenAI,
  Anthropic, Ollama, Azure, or AWS Bedrock).